Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **Optimización de hiperparámetros con *grid search* y *random search***

Vamos a trabajar con el *data set* `spambase.csv`.

El mismo es una modificación del data set publicado en el siguiente [enlace](https://www.kaggle.com/datasets/somesh24/spambase).

Este contiene registros de *emails* junto a si son clasificados como *spam* (1) o no (0). Los atributos representan la frecuencia de aparición de ciertas palabras, caracteres y mayúsculas. Para más información al respecto, visitar el enlace antes adjunto.

In [2]:
import pandas as pd

data = pd.read_csv('spambase.csv')
data.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,Class
0,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
1,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
2,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,1.85,0.00,0.00,1.85,0.00,0.00,...,0.00,0.223,0.0,0.000,0.000,0.000,3.000,15,54,1


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   word_freq_make              4600 non-null   float64
 1   word_freq_address           4600 non-null   float64
 2   word_freq_all               4600 non-null   float64
 3   word_freq_3d                4600 non-null   float64
 4   word_freq_our               4600 non-null   float64
 5   word_freq_over              4600 non-null   float64
 6   word_freq_remove            4600 non-null   float64
 7   word_freq_internet          4600 non-null   float64
 8   word_freq_order             4600 non-null   float64
 9   word_freq_mail              4600 non-null   float64
 10  word_freq_receive           4600 non-null   float64
 11  word_freq_will              4600 non-null   float64
 12  word_freq_people            4600 non-null   float64
 13  word_freq_report            4600 

In [4]:
X = data.drop('Class', axis = 'columns') # Nos quedamos con los predictores.
y = data['Class'] # Nos quedamos sólo con las etiquetas.

### ***Performance* del modelo sin optimización de hiperparámetros**

Definamos un árbol de decisión, para clasificación, con sus hiperparámetros por defecto.

Para ser consistentes con la manera de comparar el *score*, este será calculado haciendo *cross-validation*.
Para ello, scikit-learn incorpora la función `cross_val_score`.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

tree = DecisionTreeClassifier(random_state = 22) # Creamos la instancia.
scores = cross_val_score(tree, X, y, cv = KFold(4)) # Ejecutamos la validacion cruzada con 4 folds.

print(f'Score: {scores.mean():.2f}.')

Score: 0.83.


### **¿Qué hiperparámetros mejoran el score en validación?**

##### **Grid search**

Para explorar las distintas combinaciones posibles con grid search, es necesario explicitar los posibles valores que queremos que considere para cada hiperparámetro.

Los hiperparámetros que vamos a considerar son:
- `criterion`: {gini, entropy, log_loss};
- `splitter`: {best, random};
- `max_depth`: {5, 10, 20, 30, 40};
- `min_samples_split`: {2, 5, 10, 20};
- `min_samples_leaf`: {1, 2, 5, 10, 15} y
- `min_impurity_decrease`: {0, 0.05, 0.1}.

Resultan 1.800 combinaciones posibles: 3 * 2 * 5 * 4 * 5 * 3 = 1.800.

In [6]:
from sklearn.model_selection import GridSearchCV

parameters = {'criterion':('gini', 'entropy', 'log_loss'),
              'splitter': ('best', 'random'),
              'max_depth': (5, 10, 20, 30, 40),
              'min_samples_split': (2, 5, 10, 20),
              'min_samples_leaf': (1, 2, 5, 10, 15),
              'min_impurity_decrease': (0, 0.05, 0.1)
             }

# GridSearchCV recibe un estimador que debe tener implementado el método score().
# Se definen los (hiper)parámetros a combinar.
# Se pueden definir los k-folds para cross-validation. Por defecto, cv = 5.
gs = GridSearchCV(estimator = DecisionTreeClassifier(random_state = 22),
                  param_grid = parameters,
                  cv = KFold(4))

Una vez definida la instancia del `GridSearchCV`, iniciamos la búsqueda mediante el método `fit(X, y)`.

¿Corre rápidamente? ¿Lentamente? ¿Por qué?

In [7]:
gs.fit(X, y)

GridSearchCV(cv=KFold(n_splits=4, random_state=None, shuffle=False),
             estimator=DecisionTreeClassifier(random_state=22),
             param_grid={'criterion': ('gini', 'entropy', 'log_loss'),
                         'max_depth': (5, 10, 20, 30, 40),
                         'min_impurity_decrease': (0, 0.05, 0.1),
                         'min_samples_leaf': (1, 2, 5, 10, 15),
                         'min_samples_split': (2, 5, 10, 20),
                         'splitter': ('best', 'random')})

El score máximo se encuentra disponible en el atributo `best_score_`.

Los hiperparámetros que dieron con el mejor modelo se encuentran en el atributo `best_params_`.

In [8]:
print(gs.best_score_)
print(gs.best_params_)

0.8469565217391305
{'criterion': 'entropy', 'max_depth': 20, 'min_impurity_decrease': 0, 'min_samples_leaf': 5, 'min_samples_split': 20, 'splitter': 'random'}


También, podemos acceder al modelo entrenado con los hiperparámetros "ganadores" para predecir para nuevos datos o calcular el score sobre un data set de evaluación.

In [9]:
best_tree = gs.best_estimator_
best_tree

DecisionTreeClassifier(criterion='entropy', max_depth=20,
                       min_impurity_decrease=0, min_samples_leaf=5,
                       min_samples_split=20, random_state=22,
                       splitter='random')

##### **Random search**

Con esta otra estrategia, las combinaciones son seleccionadas de manera aleatoria.

La selección aleatoria es con reemplazo.

Además de definir posibles valores, también estableceremos las distribuciones de valores continuos que pueden tomar los hiperparámetros correspondientes.

Asimismo, debemos establecer el punto de corte, mediante el número de iteraciones máximas.

Los hiperparámetros que vamos a considerar y sus distribuciones son:
- `criterion`: {gini, entropy, log_loss};
- `splitter`: {best, random};
- `max_depth`: uniform[5, 40] (discreta);
- `min_samples_split`: uniform[2, 20] (discreta);
- `min_samples_leaf`: uniform[1, 20] (discreta) y
- `min_impurity_decrease`: uniform[0, 0.1] (continua).


En cuanto al número de iteraciones a probar, lo definiremos en `1.800` (la cantidad de combinaciones probadas antes con grid search).

In [10]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

parameters = {'criterion':('gini', 'entropy', 'log_loss'),
              'splitter': ('best', 'random'),
              'max_depth': list(range(5, 41)),
              'min_samples_split': list(range(2, 21)),
              'min_samples_leaf': list(range(1, 16)),
              'min_impurity_decrease': uniform(loc = 0, scale = 0.1)
             }

# RandomizedSearchCV recibe un estimador que debe tener implementado el método score().
# param_distributions es un diccionario que puede combinar conjuntos de valores explícitos y distribuciones continuas o discretas.
# Se pueden definir los k-folds para cross-validation. Por defecto, cv = 5.
rs = RandomizedSearchCV(estimator = DecisionTreeClassifier(random_state = 22),
                        param_distributions = parameters,
                        n_iter = 1800,
                        cv = KFold(4),
                        random_state = 22)

Realizamos la búsqueda con el método `fit(X, y)`.

In [11]:
rs.fit(X, y)

RandomizedSearchCV(cv=KFold(n_splits=4, random_state=None, shuffle=False),
                   estimator=DecisionTreeClassifier(random_state=22),
                   n_iter=1800,
                   param_distributions={'criterion': ('gini', 'entropy',
                                                      'log_loss'),
                                        'max_depth': [5, 6, 7, 8, 9, 10, 11, 12,
                                                      13, 14, 15, 16, 17, 18,
                                                      19, 20, 21, 22, 23, 24,
                                                      25, 26, 27, 28, 29, 30,
                                                      31, 32, 33, 34, ...],
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7a41c60cd550>,
                                        'min_samples_leaf': [1, 2, 3, 4, 5, 6,
                                                             7, 8, 9, 10, 11,
                                                             12, 13, 14, 15],
                                        'min_samples_split': [2, 3, 4, 5, 6, 7,
                                                              8, 9, 10, 11, 12,
                                                              13, 14, 15, 16,
                                                              17, 18, 19, 20],
                                        'splitter': ('best', 'random')},
                   random_state=22)

De la misma forma que ante, podemos acceder al score y a los hiperparámetros del mejor modelo encontrado.

In [12]:
print(rs.best_score_)
print(rs.best_params_)

0.8593478260869565
{'criterion': 'log_loss', 'max_depth': 29, 'min_impurity_decrease': np.float64(0.0027561663861180087), 'min_samples_leaf': 7, 'min_samples_split': 17, 'splitter': 'best'}


Notemos que, si se prueba la misma cantidad de combinaciones, random search encuentra mejores configuraciones, al explorar el espacio de manera más eficiente.

Pero... **¿Y si no disponemos de buen poder de cómputo o necesitamos un resultado mas rápidamente?**

Observemos qué sucede si a random search le acotamos el **número de iteraciones a 200**.

In [13]:
rs = RandomizedSearchCV(estimator = DecisionTreeClassifier(random_state = 22),
                        param_distributions = parameters,
                        n_iter = 200,
                        cv = KFold(4),
                        random_state = 22)

rs.fit(X,y)

print(rs.best_score_)
print(rs.best_params_)

0.8386956521739131
{'criterion': 'log_loss', 'max_depth': 24, 'min_impurity_decrease': np.float64(0.009482366243070684), 'min_samples_leaf': 11, 'min_samples_split': 7, 'splitter': 'best'}


Al asignar una menor cantidad de iteraciones, el procesamiento toma menos segundos y aún así se logra encontrar una configuración que produce un resultado superior al *default* (en este caso, por poco) y (si bien menor) cercano al de grid search (estrategia que es poco eficiente).